# Robust Value of Escalation under Distribution Shift

**Pilot run** (`PILOT = True`: `gaussian_noise` only, split seed 42) on CIFAR-10 / CIFAR-10-C, executed on a Kaggle GPU.

A cheap base model (**ResNet-18**) is paired with an ensemble of larger "monitor" models (ResNet-34, ResNet-50, DenseNet-121).
A small random-forest **value gate** decides which images are worth escalating to the monitors under a fixed escalation budget.
The gate is compared against simple confidence-based baselines, random selection and an oracle, across corruption severities.

**Contents**

1. Setup: clone repo and check GPU
2. Configuration
3. Download pretrained weights
4. Load models and inference helpers
5. Load CIFAR-10 and CIFAR-10-C
6. Run inference and verify clean accuracy
7. Analysis code (features, targets, policies, statistics)
8. Run the experiment
9. Integrity checks
10. Figures
11. Latency benchmarks
12. Single-image demo
13. Save metadata and results archive
14. Appendix: failed weight-download attempts (diagnostics only)

> **Important:** pilot results are not the full experiment. Set `PILOT = False` in Section 2 for the full run.

## 1. Setup: clone repo and check GPU

In [1]:
!git clone https://github.com/huyvnphan/PyTorch_CIFAR10.git
%cd /kaggle/working/PyTorch_CIFAR10

import sys
sys.path.insert(0, "/kaggle/working/PyTorch_CIFAR10")

import torch
from cifar10_models.resnet import resnet18, resnet34, resnet50
from cifar10_models.densenet import densenet121

print("GPU available:", torch.cuda.is_available())

Cloning into 'PyTorch_CIFAR10'...
remote: Enumerating objects: 690, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 690 (delta 52), reused 36 (delta 36), pack-reused 624 (from 1)
Receiving objects: 100% (690/690), 6.58 MiB | 18.51 MiB/s, done.
Resolving deltas: 100% (269/269), done.
/kaggle/working/PyTorch_CIFAR10
GPU available: True

## 2. Configuration
All experiment settings live in this cell. The monitor acceptance rule (`ACCEPT_MIN_AGREE`) is fixed a priori and not tuned.

In [2]:
import os, sys, time, json, random, zipfile, platform, subprocess, tarfile, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

PILOT = True                 # True: gaussian_noise only, split seed 42. Set False for the full run.
MODEL_SUITE = "huyvnphan"    # "huyvnphan": resnet18 + resnet34/resnet50/densenet121 (matches synopsis)
                             # "chenyaofo": resnet20 + resnet32/resnet44/vgg11_bn (lighter)
SPLIT_SEEDS = [42, 43, 44]
BUDGETS = [0.10, 0.20, 0.30]
PRIMARY_BUDGET = 0.20
N_GATE, N_CAL, N_EVAL = 1500, 500, 8000
N_RANDOM_DRAWS = 200
N_BOOT = 2000
BATCH_SIZE = 256
CORRUPTIONS = ["gaussian_noise", "defocus_blur", "frost", "jpeg_compression"]
SEVERITIES = [1, 2, 3, 4, 5]
ACCEPT_MIN_AGREE = 2         # accept if >= 2 of 3 monitors agree with base. Fixed a priori, not tuned.
if PILOT:
    CORRUPTIONS, SPLIT_SEEDS = ["gaussian_noise"], [42]

ON_KAGGLE = Path("/kaggle/working").exists()
ROOT = Path("/kaggle/working") if ON_KAGGLE else Path(".")
OUT = ROOT / "outputs"
CACHE = OUT / f"cache_{MODEL_SUITE}"
FIGS, TABLES = OUT / "figures", OUT / "tables"
DATA_ROOT = (Path("/kaggle/temp") if Path("/kaggle/temp").exists() else ROOT) / "data"
for p in [OUT, CACHE, FIGS, TABLES, DATA_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

SUITES = {
    "huyvnphan": dict(base="resnet18", monitors=["resnet34", "resnet50", "densenet121"],
                      mean=(0.4914, 0.4822, 0.4465), std=(0.2471, 0.2435, 0.2616),
                      expected_acc={"resnet18": 93.07, "resnet34": 93.34, "resnet50": 93.65, "densenet121": 94.06},
                      tol=0.5, source="https://github.com/huyvnphan/PyTorch_CIFAR10"),
    "chenyaofo": dict(base="resnet20", monitors=["resnet32", "resnet44", "vgg11_bn"],
                      mean=(0.4914, 0.4822, 0.4465), std=(0.2470, 0.2435, 0.2616),
                      expected_acc={"resnet20": 92.60, "resnet32": 93.53, "resnet44": 94.01, "vgg11_bn": 92.79},
                      tol=1.0, source="https://github.com/chenyaofo/pytorch-cifar-models"),
}
SUITE = SUITES[MODEL_SUITE]
BASE, MONITORS = SUITE["base"], SUITE["monitors"]
ALL_MODELS = [BASE] + MONITORS

SEED = 42
random.seed(SEED); np.random.seed(SEED)
CONDITIONS = ["clean"] + [f"{c}_s{s}" for c in CORRUPTIONS for s in SEVERITIES]
CORRUPT_CONDITIONS = [c for c in CONDITIONS if c != "clean"]
RUN_T0 = time.time()
print("PILOT:", PILOT, "| suite:", MODEL_SUITE, "| conditions:", len(CONDITIONS), "| seeds:", SPLIT_SEEDS)

PILOT: True | suite: huyvnphan | conditions: 6 | seeds: [42]

## 3. Download pretrained weights
The repository's original Box.com download link is dead (HTTP 404, see the appendix), so the weights are fetched from the
Google Drive backup linked by the repository using `gdown`.

In [3]:
!pip -q install gdown

from pathlib import Path
import zipfile
import gdown

repo = Path("/kaggle/working/PyTorch_CIFAR10")
zip_path = repo / "state_dicts.zip"
weights_dir = repo / "cifar10_models" / "state_dicts"

# Official Google Drive backup linked by the repository
file_id = "17fmN8eQdLpq2jIMQ_X0IXDPXfI9oVWgq"

result = gdown.download(
    id=file_id,
    output=str(zip_path),
    quiet=False,
    fuzzy=True
)

if not result or not zipfile.is_zipfile(zip_path):
    raise RuntimeError(
        "Download is not a valid ZIP. Check Kaggle Internet access "
        "and the Google Drive link."
    )

with zipfile.ZipFile(zip_path, "r") as z:
    bad_file = z.testzip()
    if bad_file:
        raise RuntimeError(f"Corrupted ZIP member: {bad_file}")
    z.extractall(repo / "cifar10_models")

print("\nWeight files:")
for name in ["resnet18.pt", "resnet34.pt", "resnet50.pt", "densenet121.pt"]:
    p = weights_dir / name
    print(f"{name}: {'OK' if p.exists() else 'MISSING'}",
          f"({p.stat().st_size / 1e6:.1f} MB)" if p.exists() else "")

assert all(
    (weights_dir / name).exists()
    for name in ["resnet18.pt", "resnet34.pt", "resnet50.pt", "densenet121.pt"]
), "Expected pretrained weights were not extracted."

Downloading...
From (original): https://drive.google.com/uc?id=17fmN8eQdLpq2jIMQ_X0IXDPXfI9oVWgq
From (redirected): https://drive.google.com/uc?id=17fmN8eQdLpq2jIMQ_X0IXDPXfI9oVWgq&confirm=t&uuid=82651875-42cf-491c-9c61-16609950ec98
To: /kaggle/working/PyTorch_CIFAR10/state_dicts.zip

100%|██████████| 979M/979M [00:10<00:00, 95.0MB/s]

Weight files:
resnet18.pt: OK (44.8 MB)
resnet34.pt: OK (85.3 MB)
resnet50.pt: OK (94.4 MB)
densenet121.pt: OK (28.4 MB)

## 4. Load models and inference helpers

In [4]:
import torch
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import spearmanr

torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

print("PyTorch", torch.__version__, "| device", DEVICE)

MODEL_PROVENANCE = {"suite": MODEL_SUITE, "source": SUITE["source"]}


def load_models():
    models = {}

    if MODEL_SUITE == "huyvnphan":
        repo = ROOT / "PyTorch_CIFAR10"

        if not repo.exists():
            subprocess.run(
                ["git", "clone",
                 "https://github.com/huyvnphan/PyTorch_CIFAR10.git",
                 str(repo)],
                check=True
            )

        sys.path.insert(0, str(repo))

        from cifar10_models.resnet import resnet18, resnet34, resnet50
        from cifar10_models.densenet import densenet121

        constructors = {
            "resnet18": resnet18,
            "resnet34": resnet34,
            "resnet50": resnet50,
            "densenet121": densenet121,
        }

        for name in ALL_MODELS:
            print("Loading", name)
            models[name] = constructors[name](pretrained=True).eval().to(DEVICE)

    else:
        for name in ALL_MODELS:
            print("Loading", name)
            model = torch.hub.load(
                "chenyaofo/pytorch-cifar-models",
                f"cifar10_{name}",
                pretrained=True
            )
            models[name] = model.eval().to(DEVICE)

    return models


models = load_models()
print("All models loaded successfully.")

MEAN_T = torch.tensor(SUITE["mean"]).view(1, 3, 1, 1)
STD_T = torch.tensor(SUITE["std"]).view(1, 3, 1, 1)


def preprocess_uint8(images):
    x = torch.from_numpy(
        np.ascontiguousarray(images)
    ).permute(0, 3, 1, 2).float() / 255.0

    return (x - MEAN_T) / STD_T


@torch.inference_mode()
def predict_probs(model, images, batch_size=BATCH_SIZE):
    out = []

    for start in range(0, len(images), batch_size):
        xb = preprocess_uint8(
            images[start:start + batch_size]
        ).to(DEVICE)

        out.append(
            torch.softmax(model(xb), dim=1).float().cpu().numpy()
        )

    return np.concatenate(out, axis=0)

PyTorch 2.11.0+cu128 | device cuda
Loading resnet18
Loading resnet34
Loading resnet50
Loading densenet121
All models loaded successfully.

## 5. Load CIFAR-10 and CIFAR-10-C
CIFAR-10-C is streamed from Zenodo (only the needed corruption files are kept) unless a local copy is attached.
Labels are verified against the clean test labels for every severity block.

In [5]:
from torchvision import datasets
clean_ds = datasets.CIFAR10(root=str(DATA_ROOT), train=False, download=True)
X_clean = np.asarray(clean_ds.data, dtype=np.uint8)
y_clean = np.asarray(clean_ds.targets, dtype=np.int64)
assert X_clean.shape == (10000, 32, 32, 3) and y_clean.shape == (10000,)

ZENODO_URL = "https://zenodo.org/records/2535967/files/CIFAR-10-C.tar?download=1"

def find_local_cifar_c():
    for root in [Path("/kaggle/input"), DATA_ROOT]:
        if root.exists():
            for p in root.rglob("labels.npy"):
                if (p.parent / "gaussian_noise.npy").exists():
                    return p.parent
    return None

def download_cifar_c(dest):
    """Stream the Zenodo tar and keep only the needed members (the tar is ~2.7 GB)."""
    need = {f"{c}.npy" for c in CORRUPTIONS} | {"labels.npy"}
    dest.mkdir(parents=True, exist_ok=True)
    got = {n for n in need if (dest / n).exists()}
    if got == need:
        return
    print("Streaming CIFAR-10-C from Zenodo (several minutes)...")
    with urllib.request.urlopen(ZENODO_URL) as resp, tarfile.open(fileobj=resp, mode="r|") as tar:
        for m in tar:
            name = os.path.basename(m.name)
            if m.isfile() and name in need and name not in got:
                with open(dest / (name + ".part"), "wb") as f:
                    f.write(tar.extractfile(m).read())
                (dest / (name + ".part")).replace(dest / name)
                got.add(name); print("  got", name)
            if got == need:
                break
    missing = need - got
    if missing:
        raise RuntimeError(f"CIFAR-10-C files missing: {missing}. Attach a CIFAR-10-C Kaggle dataset instead.")

CDIR = find_local_cifar_c()
if CDIR is None:
    CDIR = DATA_ROOT / "CIFAR-10-C"
    download_cifar_c(CDIR)
print("CIFAR-10-C dir:", CDIR)
labels_c = np.load(CDIR / "labels.npy")
cifar_c = {c: np.load(CDIR / f"{c}.npy", mmap_mode="r") for c in CORRUPTIONS}
for c, arr in cifar_c.items():
    assert arr.shape == (50000, 32, 32, 3), (c, arr.shape)
# Integrity: every severity block carries the clean test labels in the same order.
assert labels_c.shape == (50000,) and all(
    np.array_equal(labels_c[(s - 1) * 10000: s * 10000], y_clean) for s in SEVERITIES), "label blocks mismatch"
print("Label blocks verified against clean test labels.")

100%|██████████| 170M/170M [27:23<00:00, 104kB/s]

Streaming CIFAR-10-C from Zenodo (several minutes)...
  got gaussian_noise.npy
  got labels.npy
CIFAR-10-C dir: /kaggle/working/data/CIFAR-10-C
Label blocks verified against clean test labels.

## 6. Run inference and verify clean accuracy
Softmax probabilities for all four models are cached per condition. Clean accuracy is checked against the published numbers
as a guard against wrong weights or preprocessing.

In [6]:
def condition_images(cond):
    if cond == "clean":
        return X_clean
    corr, sev = cond.rsplit("_s", 1)
    sev = int(sev)
    return cifar_c[corr][(sev - 1) * 10000: sev * 10000]

def infer_condition(cond):
    f = CACHE / f"{cond}.npz"
    if f.exists():
        z = np.load(f)
        return {k: z[k] for k in z.files}
    X = np.asarray(condition_images(cond))
    res = {"y": y_clean}
    for n in ALL_MODELS:
        res[n] = predict_probs(models[n], X).astype(np.float32)
    tmp = CACHE / f"{cond}.tmp.npz"
    np.savez_compressed(tmp, **res)
    tmp.replace(f)
    return res

t0 = time.time()
PROBS = {}
for i, cond in enumerate(CONDITIONS, 1):
    PROBS[cond] = infer_condition(cond)
    print(f"[{i}/{len(CONDITIONS)}] {cond}", end="\r")
INFER_MINUTES = (time.time() - t0) / 60
print(f"\nInference/cache load: {INFER_MINUTES:.1f} min")

# Clean accuracy vs published numbers (tight check on all 10,000 images)
CLEAN_ACC = {n: float((PROBS["clean"][n].argmax(1) == y_clean).mean() * 100) for n in ALL_MODELS}
for n in ALL_MODELS:
    exp = SUITE["expected_acc"][n]
    print(f"  {n:12s} clean acc {CLEAN_ACC[n]:.2f}%  (published {exp:.2f}%)")
    assert abs(CLEAN_ACC[n] - exp) <= SUITE["tol"], (
        f"{n}: {CLEAN_ACC[n]:.2f}% vs published {exp}% -> preprocessing or weights are wrong")

# ===ANALYSIS_START (pure numpy/sklearn: no torch needed)

/tmp/ipykernel_48/1843819487.py:67: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  x = torch.from_numpy(

[6/6] gaussian_noise_s5
Inference/cache load: 1.1 min
  resnet18     clean acc 93.07%  (published 93.07%)
  resnet34     clean acc 93.33%  (published 93.34%)
  resnet50     clean acc 93.65%  (published 93.65%)
  densenet121  clean acc 94.06%  (published 94.06%)

## 7. Analysis code
Pure NumPy / scikit-learn from here on (no torch needed).

### 7.1 Features, value target and gate fitting
* **Features** from the base model's softmax: max confidence, entropy, top-2 margin.
* **Value target** `V* = u1 - u0 ∈ {-1, 0, +1}`: `+1` the monitor caught a base error, `-1` it withheld a correct answer, `0` escalation was wasted.
* **Splits**: gate-fit / calibration / evaluation, disjoint, all drawn from the CIFAR-10 test set.

In [7]:
def cheap_features(probs):
    p = np.clip(probs, 1e-12, 1.0)
    sp = np.sort(p, axis=1)
    conf = sp[:, -1]
    margin = sp[:, -1] - sp[:, -2]
    ent = -(p * np.log(p)).sum(axis=1)
    return np.column_stack([conf, ent, margin]).astype(np.float32)

FEATURE_NAMES = ["max_confidence", "entropy", "top2_margin"]

def value_target(P, y):
    """u0 = base utility (+1 correct, -1 wrong); u1 = utility after monitor (served as u0 if accepted, else abstain = 0).
    V* = u1 - u0 in {-1, 0, +1}: -1 harmful (correct answer withheld), +1 useful (error caught), 0 wasted."""
    base_pred = P[BASE].argmax(1)
    votes = np.column_stack([P[m].argmax(1) for m in MONITORS])
    accepted = (votes == base_pred[:, None]).sum(1) >= ACCEPT_MIN_AGREE
    u0 = np.where(base_pred == y, 1.0, -1.0)
    u1 = np.where(accepted, u0, 0.0)
    return u1 - u0, u0, accepted

def make_splits(seed):
    perm = np.random.default_rng(seed).permutation(10000)
    g, c, e = perm[:N_GATE], perm[N_GATE:N_GATE + N_CAL], perm[N_GATE + N_CAL:N_GATE + N_CAL + N_EVAL]
    assert not (set(g) & set(c)) and not (set(g) & set(e)) and not (set(c) & set(e))
    assert len(g) == N_GATE and len(c) == N_CAL and len(e) == N_EVAL
    return dict(gate=g, cal=c, eval=e)

def fit_gates(clean_P, y, gate_idx, seed):
    P = {k: clean_P[k][gate_idx] for k in ALL_MODELS}
    v, _, _ = value_target(P, y[gate_idx])
    X = cheap_features(P[BASE])
    full = RandomForestRegressor(n_estimators=200, min_samples_leaf=10, random_state=seed, n_jobs=-1).fit(X, v)
    ent = RandomForestRegressor(n_estimators=200, min_samples_leaf=10, random_state=seed, n_jobs=-1).fit(X[:, [1]], v)
    return full, ent, v

### 7.2 Escalation policies and metrics
Policies compared at each budget: `value_gate`, `value_gate_entropy_only` (ablation), `low_confidence`, `high_entropy`, `low_margin`,
`random`, `oracle`, `direct_abstain_matched`, plus fixed clean-calibrated thresholds (`*_fixed_thr`) to measure escalation-rate drift under shift.

In [8]:
def topk_mask(scores, budget):
    n = len(scores)
    k = max(1, int(round(budget * n)))
    order = np.lexsort((np.arange(n), -np.asarray(scores, dtype=np.float64)))   # deterministic tie-breaking
    m = np.zeros(n, bool); m[order[:k]] = True
    return m

def rank_corr(s, v):
    if np.std(s) == 0 or np.std(v) == 0:
        return np.nan          # undefined: report as undefined, not zero
    return float(spearmanr(s, v).statistic)

def esc_metrics(mask, ctx):
    v, be, acc, u0 = ctx["v"], ctx["be"], ctx["acc"], ctx["u0"]
    abstain = mask & ~acc
    served = ~abstain
    return dict(escalation_rate=float(mask.mean()), utility=float(np.mean(u0 + mask * v)),
                gain=float(np.mean(mask * v)), coverage=float(served.mean()),
                error_rate_served=float(be[served].mean()) if served.any() else np.nan,
                ensemble_calls_per_image=float(mask.mean() * len(MONITORS)),
                selected_error_rate=float(be[mask].mean()) if mask.any() else np.nan,
                selected_positive_value_rate=float((v[mask] > 0).mean()) if mask.any() else np.nan,
                missed_error_rate=float(np.mean(be & ~mask)))

def abstain_metrics(abstain, ctx):
    be, u0 = ctx["be"], ctx["u0"]
    served = ~abstain
    u = np.where(abstain, 0.0, u0)
    return dict(escalation_rate=0.0, utility=float(u.mean()), gain=float(np.mean(u - u0)),
                coverage=float(served.mean()),
                error_rate_served=float(be[served].mean()) if served.any() else np.nan,
                ensemble_calls_per_image=0.0, selected_error_rate=np.nan,
                selected_positive_value_rate=np.nan, missed_error_rate=np.nan)

def parse_cond(c):
    if c == "clean":
        return "clean", 0
    a, b = c.rsplit("_s", 1)
    return a, int(b)

def run_seed(seed, probs_by_cond, y, conditions):
    """Fit gates on the clean gate-fit split, calibrate thresholds on the clean calibration split,
    evaluate every condition on the disjoint eval split. Returns result rows + per-example utilities."""
    sp = make_splits(seed)
    gate, ent_gate, v_fit = fit_gates(probs_by_cond["clean"], y, sp["gate"], seed)
    cal_feats = cheap_features(probs_by_cond["clean"][BASE][sp["cal"]])
    cal_gate, cal_conf = gate.predict(cal_feats), -cal_feats[:, 0]
    thr_gate = {b: float(np.quantile(cal_gate, 1 - b, method="higher")) for b in BUDGETS}
    thr_conf = {b: float(np.quantile(cal_conf, 1 - b, method="higher")) for b in BUDGETS}
    ev = sp["eval"]; y_ev = y[ev]
    rows, U = [], {}
    for ci, cond in enumerate(conditions):
        P = {k: probs_by_cond[cond][k][ev] for k in ALL_MODELS}
        v, u0, acc = value_target(P, y_ev)
        ctx = dict(v=v, u0=u0, acc=acc, be=(u0 < 0))
        feats = cheap_features(P[BASE])
        scores = {"value_gate": gate.predict(feats),
                  "value_gate_entropy_only": ent_gate.predict(feats[:, [1]]),
                  "low_confidence": -feats[:, 0], "high_entropy": feats[:, 1], "low_margin": -feats[:, 2]}
        corr, sev = parse_cond(cond)
        base = dict(condition=cond, corruption=corr, severity=sev, seed=seed, n=len(y_ev),
                    base_error_rate=float(ctx["be"].mean()), mean_v=float(v.mean()),
                    frac_v_pos=float((v > 0).mean()), frac_v_neg=float((v < 0).mean()))
        def add(policy, budget, met, spearman=np.nan):
            rows.append({**base, "policy": policy, "budget": budget, "rank_spearman": spearman, **met})
        # budget-free bounds
        add("cheap_only", 0.0, esc_metrics(np.zeros(len(v), bool), ctx))
        add("always_on", 1.0, esc_metrics(np.ones(len(v), bool), ctx))
        for b in BUDGETS:
            masks = {}
            for pol, s in scores.items():
                masks[pol] = topk_mask(s, b)
                add(pol, b, esc_metrics(masks[pol], ctx), rank_corr(s, v))
            # random: average over draws
            rng = np.random.default_rng([seed, ci, int(round(b * 100))])
            draws = []
            for _ in range(N_RANDOM_DRAWS):
                m = np.zeros(len(v), bool)
                m[rng.choice(len(v), size=max(1, int(round(b * len(v)))), replace=False)] = True
                draws.append(esc_metrics(m, ctx))
            add("random", b, {k: float(np.nanmean([d[k] for d in draws])) for k in draws[0]})
            # oracle: best fixed-budget escalation set for THIS monitor (not an upper bound for direct abstention)
            masks["oracle"] = topk_mask(v, b)
            add("oracle", b, esc_metrics(masks["oracle"], ctx))
            # direct abstention matched to the value gate's coverage (abstain on lowest-confidence images)
            n_abs = int((masks["value_gate"] & ~acc).sum())
            da = np.zeros(len(v), bool)
            if n_abs > 0:
                da[np.argsort(feats[:, 0], kind="stable")[:n_abs]] = True
            add("direct_abstain_matched", b, abstain_metrics(da, ctx))
            # fixed (clean-calibrated) thresholds: escalation rate may drift under shift
            add("value_gate_fixed_thr", b, esc_metrics(scores["value_gate"] >= thr_gate[b], ctx))
            add("low_confidence_fixed_thr", b, esc_metrics(scores["low_confidence"] >= thr_conf[b], ctx))
            if abs(b - PRIMARY_BUDGET) < 1e-9:      # per-example utilities for the paired bootstrap
                U[cond] = {"value_gate": u0 + masks["value_gate"] * v,
                           "low_confidence": u0 + masks["low_confidence"] * v,
                           "value_gate_entropy_only": u0 + masks["value_gate_entropy_only"] * v,
                           "direct_abstain_matched": np.where(da, 0.0, u0)}
    return pd.DataFrame(rows), U, dict(split=sp, gate=gate, ent_gate=ent_gate, thr_gate=thr_gate,
                                       thr_conf=thr_conf, v_fit=v_fit)

### 7.3 Statistics: paired bootstrap, Holm correction and verdict

In [9]:
def holm(p):
    p = np.asarray(p, float); m = len(p); adj = np.empty(m); run = 0.0
    for r, i in enumerate(np.argsort(p)):
        run = max(run, (m - r) * p[i]); adj[i] = min(1.0, run)
    return adj

def bootstrap_pair(U_a, U_b, conds, seed, n_boot=N_BOOT, chunk=50):
    D = np.stack([U_a[c] - U_b[c] for c in conds]).astype(np.float32)   # base utility cancels in the paired difference
    C, n = D.shape
    rng = np.random.default_rng(seed + 999)
    boots = np.empty((C, n_boot), np.float32)
    for s in range(0, n_boot, chunk):
        m = min(chunk, n_boot - s)
        idx = rng.integers(0, n, size=(m, n))          # same resampled images for all conditions
        boots[:, s:s + m] = D[:, idx].mean(axis=2)
    return D.mean(axis=1), boots

def p_two_sided(b):
    return min(1.0, 2 * (min(int((b <= 0).sum()), int((b >= 0).sum())) + 1) / (len(b) + 1))

def bootstrap_tables(U, seed, a, b, conds):
    point, boots = bootstrap_pair({c: U[c][a] for c in conds}, {c: U[c][b] for c in conds}, conds, seed)
    rows = []
    for i, c in enumerate(conds):
        lo, hi = np.quantile(boots[i], [0.025, 0.975])
        rows.append(dict(seed=seed, comparison=f"{a} - {b}", condition=c, diff=float(point[i]),
                         ci_low=float(lo), ci_high=float(hi), p=p_two_sided(boots[i])))
    t = pd.DataFrame(rows)
    corrupt = t["condition"] != "clean"
    t.loc[corrupt, "p_holm"] = holm(t.loc[corrupt, "p"].values)     # per-condition tests, corrupted only
    ci = [i for i, c in enumerate(conds) if c != "clean"]
    prim_b = boots[ci].mean(axis=0)
    prim = dict(seed=seed, comparison=f"{a} - {b}", budget=PRIMARY_BUDGET,
                n_corrupted_conditions=len(ci), diff=float(point[ci].mean()),
                ci_low=float(np.quantile(prim_b, 0.025)), ci_high=float(np.quantile(prim_b, 0.975)),
                p=p_two_sided(prim_b))
    return t, prim

def verdict(prim_df):
    d = prim_df[prim_df["comparison"] == "value_gate - low_confidence"]
    pos = (d["diff"] > 0).all(); excl = ((d["ci_low"] > 0) | (d["ci_high"] < 0)).all()
    if pos and (d["ci_low"] > 0).all():
        return "GATE BEATS CONFIDENCE (positive, CI excludes 0, sign consistent across all seeds)"
    if (d["diff"] < 0).all() and (d["ci_high"] < 0).all():
        return "CONFIDENCE BEATS GATE (negative, CI excludes 0 in all seeds)"
    return "MIXED / NO ADVANTAGE (non-significance is not proof of equivalence)"
# ===ANALYSIS_END

## 8. Run the experiment
For each split seed: fit the gates on clean data, evaluate every condition, and run the paired bootstrap.
The primary endpoint is `value_gate - low_confidence` at the 20% budget, averaged over corrupted conditions.

In [10]:
def show(df):
    try:
        display(df)
    except NameError:
        print(df.to_string(max_rows=30))

all_rows, boot_tables, prim_rows, RUN = [], [], [], {}
for seed in SPLIT_SEEDS:
    print(f"=== split seed {seed} ===")
    df, U, art = run_seed(seed, PROBS, y_clean, CONDITIONS)
    all_rows.append(df); RUN[seed] = art
    print("  V* on gate-fit clean split:", pd.Series(art["v_fit"]).value_counts().to_dict(),
          "| gate importances:", dict(zip(FEATURE_NAMES, np.round(art["gate"].feature_importances_, 3))))
    for a, b in [("value_gate", "low_confidence"), ("value_gate", "value_gate_entropy_only"),
                 ("value_gate", "direct_abstain_matched")]:
        t, pr = bootstrap_tables(U, seed, a, b, CONDITIONS)
        boot_tables.append(t); prim_rows.append(pr)
    np.savez(TABLES / f"split_indices_seed{seed}.npz", **art["split"])

results = pd.concat(all_rows, ignore_index=True)
results.to_csv(TABLES / "all_policy_results.csv", index=False)
boot_df = pd.concat(boot_tables, ignore_index=True); boot_df.to_csv(TABLES / "bootstrap_per_condition.csv", index=False)
prim_df = pd.DataFrame(prim_rows); prim_df.to_csv(TABLES / "primary_endpoint.csv", index=False)

num = ["gain", "utility", "escalation_rate", "coverage", "error_rate_served", "ensemble_calls_per_image",
       "selected_error_rate", "rank_spearman", "base_error_rate"]
summary = results.groupby(["condition", "budget", "policy"], dropna=False)[num].agg(["mean", "std"])
summary.columns = ["_".join(c) for c in summary.columns]
summary = summary.reset_index(); summary.to_csv(TABLES / "summary_by_condition.csv", index=False)
corrupt = results[results["condition"] != "clean"]
overall = (corrupt.groupby(["budget", "policy"])[num].mean().reset_index())
overall.to_csv(TABLES / "overall_corrupted_by_policy.csv", index=False)
clean_tab = (results[results["condition"] == "clean"].groupby(["budget", "policy"])[num].mean().reset_index())
clean_tab.to_csv(TABLES / "clean_by_policy.csv", index=False)
abl = (corrupt[(corrupt["budget"] == PRIMARY_BUDGET) & corrupt["policy"].isin(["value_gate", "value_gate_entropy_only"])]
       .groupby("policy")[["gain", "rank_spearman"]].mean())
abl.to_csv(TABLES / "feature_ablation.csv")
drift = results[results["policy"].isin(["value_gate_fixed_thr", "low_confidence_fixed_thr"])][
    ["condition", "corruption", "severity", "seed", "budget", "policy", "escalation_rate"]]
drift.to_csv(TABLES / "threshold_rate_drift.csv", index=False)

print("\nPRIMARY ENDPOINT (value_gate - low_confidence, 20% budget, mean over corrupted conditions):")
show(prim_df[prim_df["comparison"] == "value_gate - low_confidence"])
print("VERDICT:", verdict(prim_df))
print("\nCorrupted conditions, 20% budget, mean over conditions and seeds:")
show(overall[overall["budget"].isin([PRIMARY_BUDGET, 0.0, 1.0])].round(4))
print("\nAblation (3-feature vs entropy-only):"); show(abl.round(4))

=== split seed 42 ===
  V* on gate-fit clean split: {0.0: 1430, 1.0: 46, -1.0: 24} | gate importances: {'max_confidence': np.float64(0.339), 'entropy': np.float64(0.398), 'top2_margin': np.float64(0.263)}

PRIMARY ENDPOINT (value_gate - low_confidence, 20% budget, mean over corrupted conditions):

   seed                   comparison  budget  n_corrupted_conditions     diff    ci_low   ci_high      p
0    42  value_gate - low_confidence     0.2                       5 -0.00935 -0.011975 -0.006874  0.001

VERDICT: CONFIDENCE BEATS GATE (negative, CI excludes 0 in all seeds)

Corrupted conditions, 20% budget, mean over conditions and seeds:

    budget                    policy    gain  utility  escalation_rate  coverage  error_rate_served  ensemble_calls_per_image  selected_error_rate  rank_spearman  base_error_rate
0      0.0                cheap_only  0.0000   0.1620           0.0000    1.0000             0.4190                    0.0000                  NaN            NaN            0.419
11     0.2    direct_abstain_matched  0.0426   0.2046           0.0000    0.8981             0.3891                    0.0000                  NaN            NaN            0.419
12     0.2              high_entropy  0.0534   0.2154           0.2000    0.8858             0.3808                    0.6000               0.6675         0.2161            0.419
13     0.2            low_confidence  0.0548   0.2168           0.2000    0.8857             0.3799                    0.6000               0.6642         0.2162            0.419
14     0.2  low_confidence_fixed_thr  0.0871   0.2491           0.6082    0.7701             0.3555      

Ablation (3-feature vs entropy-only):

                           gain  rank_spearman
policy                                        
value_gate               0.0455         0.1278
value_gate_entropy_only  0.0449         0.1094

**Pilot reading:** on `gaussian_noise` (5 severities), the plain **low-confidence** baseline beats the learned value gate at the 20% budget
(difference −0.0094 utility, 95% CI [−0.0120, −0.0069]). The gate was trained on clean data, where `V* = ±1` is rare (70 of 1,500 examples),
so its ranking degrades under noise. Treat this as a pilot result only, not as the full experiment.

## 9. Integrity checks

In [11]:
def run_integrity_checks():
    for seed, art in RUN.items():
        sp = art["split"]
        assert len(set(sp["gate"]) | set(sp["cal"]) | set(sp["eval"])) == 10000, "splits must partition the test set"
    assert set(np.unique(np.concatenate([np.round(RUN[s]["v_fit"]) for s in RUN]))) <= {-1.0, 0.0, 1.0}
    for cond in CONDITIONS:
        for m in ALL_MODELS:
            assert PROBS[cond][m].shape == (10000, 10)
            assert np.allclose(PROBS[cond][m].sum(1), 1, atol=1e-3), (cond, m)
    r = results
    for b in BUDGETS:
        for pol in ["value_gate", "low_confidence", "high_entropy", "low_margin", "oracle", "value_gate_entropy_only"]:
            er = r[(r.policy == pol) & (r.budget == b)]["escalation_rate"]
            assert np.allclose(er, round(b * N_EVAL) / N_EVAL), (pol, b)
        assert np.allclose(r[(r.policy == "random") & (r.budget == b)]["escalation_rate"], b, atol=1e-3)
    g = r[(r.policy == "always_on")]["gain"].values; assert np.isfinite(g).all()
    assert (r[r.policy == "oracle"]["gain"].values + 1e-9 >=
            r[r.policy == "value_gate"]["gain"].values).all(), "oracle must dominate the gate among escalation policies"
    # bootstrap point estimate must equal the difference of gains
    for seed in SPLIT_SEEDS:
        a = r[(r.seed == seed) & (r.budget == PRIMARY_BUDGET) & (r.policy == "value_gate") & (r.condition != "clean")]["gain"].mean()
        b_ = r[(r.seed == seed) & (r.budget == PRIMARY_BUDGET) & (r.policy == "low_confidence") & (r.condition != "clean")]["gain"].mean()
        pe = prim_df[(prim_df.seed == seed) & (prim_df.comparison == "value_gate - low_confidence")]["diff"].iloc[0]
        assert abs((a - b_) - pe) < 1e-6, "primary endpoint mismatch"
    base = r[(r.policy == "cheap_only") & (r.budget == 0.0)]
    cl = base[base.condition == "clean"]["base_error_rate"].mean()
    s5 = base[base.severity == 5]["base_error_rate"].mean()
    assert s5 > cl, "severity-5 error should exceed clean error"
    print("ALL INTEGRITY CHECKS PASSED")
run_integrity_checks()

ALL INTEGRITY CHECKS PASSED

## 10. Figures
Figures are saved to `outputs/figures/` (`utility_gain_20pct.png`, `spearman_by_severity.png`, `threshold_rate_drift.png`;
`gain_by_severity.png` is only produced when `PILOT = False`). Plot images were not captured in the exported outputs,
so re-run this cell to regenerate them.

In [12]:
def savefig(name):
    plt.tight_layout(); plt.savefig(FIGS / name, dpi=200); plt.show()

pol_order = ["oracle", "value_gate", "value_gate_entropy_only", "low_confidence", "high_entropy", "low_margin", "random"]
o20 = overall[overall["budget"] == PRIMARY_BUDGET].set_index("policy").reindex(pol_order)
plt.figure(figsize=(9, 5)); o20["gain"].plot(kind="bar"); plt.ylabel("Mean utility gain over cheap-only")
plt.title("Utility gain at 20% budget (corrupted conditions, mean of seeds)"); plt.xticks(rotation=30, ha="right")
savefig("utility_gain_20pct.png")

if not PILOT:
    plt.figure(figsize=(9, 5))
    for pol in ["value_gate", "low_confidence"]:
        d = corrupt[(corrupt.budget == PRIMARY_BUDGET) & (corrupt.policy == pol)].groupby("severity")["gain"].mean()
        plt.plot(d.index, d.values, marker="o", label=pol)
    plt.xlabel("Severity"); plt.ylabel("Utility gain"); plt.legend(); plt.title("Gate vs confidence across severity")
    savefig("gain_by_severity.png")

for corr in CORRUPTIONS:
    pass
plt.figure(figsize=(9, 5))
for corr, g in corrupt[(corrupt.budget == PRIMARY_BUDGET) & (corrupt.policy == "value_gate")].groupby("corruption"):
    d = g.groupby("severity")["rank_spearman"].mean(); plt.plot(d.index, d.values, marker="o", label=corr)
plt.xlabel("Severity"); plt.ylabel("Spearman (gate score vs V*)"); plt.legend()
plt.title("Rank quality vs severity (read alongside utility; V* is mostly 0)")
savefig("spearman_by_severity.png")

plt.figure(figsize=(9, 5))
for pol, g in drift[(drift.budget == PRIMARY_BUDGET) & (drift.condition != "clean")].groupby("policy"):
    d = g.groupby("severity")["escalation_rate"].mean(); plt.plot(d.index, d.values, marker="o", label=pol)
plt.axhline(PRIMARY_BUDGET, ls="--", c="k"); plt.xlabel("Severity"); plt.ylabel("Actual escalation rate")
plt.title("Fixed clean-calibrated threshold: escalation-rate drift"); plt.legend()
savefig("threshold_rate_drift.png")

## 11. Latency benchmarks
Component-level timings for 2,000 images (GPU), plus end-to-end pipelines: base only, always-on ensemble, and 20% selective escalation.

In [13]:
def sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

def time_fn(fn, repeats=7):
    fn(); sync()
    ts = []
    for _ in range(repeats):
        t = time.perf_counter(); fn(); sync(); ts.append(time.perf_counter() - t)
    return float(np.mean(ts)), float(np.quantile(ts, 0.95))

N_LAT = 2000
X_lat = X_clean[:N_LAT]
seed0 = SPLIT_SEEDS[0]; gate0 = RUN[seed0]["gate"]
lat = []
for n in ALL_MODELS:
    m, p = time_fn(lambda n=n: predict_probs(models[n], X_lat))
    lat.append(dict(component=n, mean_s=m, p95_s=p, ms_per_image=1000 * m / N_LAT))
feats_lat = cheap_features(PROBS["clean"][BASE][:N_LAT])
m, p = time_fn(lambda: gate0.predict(feats_lat)); lat.append(dict(component="rf_gate", mean_s=m, p95_s=p, ms_per_image=1000 * m / N_LAT))

def pipeline(mode, budget=PRIMARY_BUDGET):
    pb = predict_probs(models[BASE], X_lat)
    if mode == "cheap_only":
        return
    if mode == "always_on":
        for mm in MONITORS: predict_probs(models[mm], X_lat)
        return
    sel = topk_mask(gate0.predict(cheap_features(pb)), budget)
    for mm in MONITORS: predict_probs(models[mm], X_lat[sel])
for mode in ["cheap_only", "always_on", "selective_20pct"]:
    m, p = time_fn(lambda mode=mode: pipeline("selective" if mode.startswith("selective") else mode))
    lat.append(dict(component=f"pipeline_{mode}", mean_s=m, p95_s=p, ms_per_image=1000 * m / N_LAT))
lat_df = pd.DataFrame(lat); lat_df.to_csv(TABLES / "latency_benchmarks.csv", index=False)
show(lat_df.round(4))

                  component  mean_s   p95_s  ms_per_image
0                  resnet18  0.2964  0.2994        0.1482
1                  resnet34  0.5108  0.5138        0.2554
2                  resnet50  0.6849  0.6895        0.3424
3               densenet121  0.6375  0.6419        0.3187
4                   rf_gate  0.0694  0.0759        0.0347
5       pipeline_cheap_only  0.3018  0.3047        0.1509
6        pipeline_always_on  2.2048  2.2188        1.1024
7  pipeline_selective_20pct  0.7523  0.7606        0.3761

## 12. Single-image demo
Walks one evaluation-split image through the full decision logic: base prediction → gate score → escalate or not → monitor vote.

In [14]:
def demo_one(condition="gaussian_noise_s3", index=0, seed=None):
    seed = seed or SPLIT_SEEDS[0]
    art = RUN[seed]
    i = int(art["split"]["eval"][int(index) % N_EVAL])           # always an eval-split image
    pr = {k: PROBS[condition][k][i:i + 1] for k in ALL_MODELS}
    f = cheap_features(pr[BASE]); score = float(art["gate"].predict(f)[0])
    base_pred = int(pr[BASE].argmax(1)[0]); votes = {m: int(pr[m].argmax(1)[0]) for m in MONITORS}
    escalate = score >= art["thr_gate"][PRIMARY_BUDGET]
    agree = sum(v == base_pred for v in votes.values())
    decision = "serve base (no escalation)" if not escalate else (
        "monitor accepts -> serve base" if agree >= ACCEPT_MIN_AGREE else "monitor rejects -> ABSTAIN")
    info = dict(condition=condition, image=i, true=int(y_clean[i]), base_pred=base_pred,
                confidence=round(float(f[0, 0]), 3), entropy=round(float(f[0, 1]), 3), margin=round(float(f[0, 2]), 3),
                gate_value=round(score, 4), escalate=bool(escalate), monitor_votes=votes, decision=decision)
    print(info)
    plt.figure(figsize=(2.5, 2.5)); plt.imshow(np.asarray(condition_images(condition)[i])); plt.axis("off")
    plt.title(decision, fontsize=8); plt.show()
    return info
_ = demo_one("gaussian_noise_s3" if "gaussian_noise_s3" in CONDITIONS else CONDITIONS[-1], 0)

{'condition': 'gaussian_noise_s3', 'image': 6910, 'true': 7, 'base_pred': 4, 'confidence': 0.658, 'entropy': 1.198, 'margin': 0.487, 'gate_value': 0.319, 'escalate': True, 'monitor_votes': {'resnet34': 7, 'resnet50': 7, 'densenet121': 7}, 'decision': 'monitor rejects -> ABSTAIN'}

## 13. Save metadata and results archive

In [15]:
meta = dict(project="Robust Value of Escalation under Distribution Shift", pilot=PILOT,
            model_suite=MODEL_SUITE, base=BASE, monitors=MONITORS, model_provenance=MODEL_PROVENANCE,
            clean_accuracy_pct=CLEAN_ACC, published_accuracy_pct=SUITE["expected_acc"],
            monitor_rule=f">= {ACCEPT_MIN_AGREE} of {len(MONITORS)} monitors agree with base (fixed a priori)",
            splits=dict(gate=N_GATE, calibration=N_CAL, eval=N_EVAL), split_seeds=SPLIT_SEEDS,
            seeds_note="split seeds only; networks are pretrained and fixed",
            corruptions=CORRUPTIONS, severities=SEVERITIES, budgets=BUDGETS, primary_budget=PRIMARY_BUDGET,
            random_draws=N_RANDOM_DRAWS, bootstrap_replicates=N_BOOT, device=str(DEVICE),
            gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            torch=torch.__version__, python=sys.version, platform=platform.platform(),
            inference_minutes=round(INFER_MINUTES, 2), total_minutes=round((time.time() - RUN_T0) / 60, 2),
            verdict=verdict(prim_df),
            deviations=["Pretrained CIFAR-10 models (not trained here)", "Gate fit/calibration/eval drawn from the CIFAR-10 test set "
                        "because the pretrained models saw the training set"])
(OUT / "run_metadata.json").write_text(json.dumps(meta, indent=2, default=str))
(OUT / "RUN_LOG.md").write_text(
    f"# RUN LOG\n- date: {time.strftime('%Y-%m-%d %H:%M')}\n- pilot: {PILOT}\n- suite: {MODEL_SUITE}\n"
    f"- runtime (min): {meta['total_minutes']}\n- errors / model-name substitutions: (fill in)\n")
zpath = ROOT / "results.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file() and CACHE not in p.parents:
            z.write(p, arcname=str(p.relative_to(OUT)))
print("Wrote", zpath, "| tables:", sorted(p.name for p in TABLES.iterdir()))
print("PILOT RESULTS ARE NOT THE FULL EXPERIMENT." if PILOT else "Full run complete.")

Wrote /kaggle/working/results.zip | tables: ['all_policy_results.csv', 'bootstrap_per_condition.csv', 'clean_by_policy.csv', 'feature_ablation.csv', 'latency_benchmarks.csv', 'overall_corrupted_by_policy.csv', 'primary_endpoint.csv', 'split_indices_seed42.npz', 'summary_by_condition.csv', 'threshold_rate_drift.csv']
PILOT RESULTS ARE NOT THE FULL EXPERIMENT.

### Reporting notes (honesty checklist)

* Every reported number must come from the saved CSVs.
* "Seeds" are **split seeds** only; the pretrained networks are fixed.
* The gate / calibration / evaluation split lives **inside the CIFAR-10 test set**, because the pretrained models saw the training set.
* The monitor acceptance rule was fixed a priori and not tuned.
* The bootstrap is conditional on the fitted gate.
* Batch top-k selection is not a streaming policy.
* Latency numbers are component-level.

In [16]:
# Report honesty: every number must come from these CSVs; seeds are split seeds with fixed pretrained models;
# the gate/calibration/eval split lives inside the test set (models saw the train set); monitor rule untuned;
# bootstrap is conditional on the fitted gate; batch top-k is not streaming; latency is component-level.

---
## 14. Appendix: failed weight-download attempts (diagnostics only)

These cells are **not needed** to reproduce the results. They document why the weights are fetched from Google Drive in Section 3:
the repository's `train.py --download_weights 1` points to a Box.com link that now returns HTTP 404 (an HTML page, not a ZIP),
which makes `zipfile` fail with `BadZipFile`. Cells are kept in their original order and are intentionally left unnumbered.

### A.1 Patch obsolete `pytorch_lightning` import

In [ ]:
from pathlib import Path

repo = Path("/kaggle/working/PyTorch_CIFAR10")
module_file = repo / "module.py"

text = module_file.read_text()
old = "from pytorch_lightning.metrics import Accuracy"

if old in text:
    module_file.write_text(text.replace(old, "# Removed obsolete Accuracy import; inference does not need it."))
    print("Compatibility patch applied.")
else:
    print("Old import not found; inspect module.py before proceeding.")

Compatibility patch applied.

### A.2 Repository's own weight downloader (fails)

In [ ]:
import subprocess
import sys
from pathlib import Path

repo = Path("/kaggle/working/PyTorch_CIFAR10")

# Install a compatible downloader
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "gdown"],
    check=True
)

# Use the repository's official weight-download script
# after patching its obsolete import
module_file = repo / "module.py"
if module_file.exists():
    txt = module_file.read_text()
    txt = txt.replace(
        "from pytorch_lightning.metrics import Accuracy",
        "# Removed obsolete metric import for weight download"
    )
    module_file.write_text(txt)

result = subprocess.run(
    [sys.executable, "train.py", "--download_weights", "1"],
    cwd=repo
)

print("Downloader exit code:", result.returncode)
print("Weight files:", list((repo / "cifar10_models" / "state_dicts").glob("*.pt"))
      if (repo / "cifar10_models" / "state_dicts").exists() else "Directory missing")

2.19kMiB [00:00, 4.72MMiB/s]
Traceback (most recent call last):
  File "/kaggle/working/PyTorch_CIFAR10/train.py", line 82, in <module>
    main(args)
    ~~~~^^^^^^
  File "/kaggle/working/PyTorch_CIFAR10/train.py", line 16, in main
    CIFAR10Data.download_weights()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/kaggle/working/PyTorch_CIFAR10/data.py", line 44, in download_weights
    with zipfile.ZipFile(path_to_zip_file, "r") as zip_ref:
         ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/zipfile/__init__.py", line 1407, in __init__
    self._RealGetContents()
    ~~~~~~~~~~~~~~~~~~~~~^^
  File "/usr/lib/python3.13/zipfile/__init__.py", line 1474, in _RealGetContents
    raise BadZipFile("File is not a zip file")
zipfile.BadZipFile: File is not a zip file

Download successful. Unzipping file...
Downloader exit code: 1
Weight files: Directory missing

### A.3 Inspect the downloader source

In [ ]:
from pathlib import Path

repo = Path("/kaggle/working/PyTorch_CIFAR10")

print("=== data.py: weight downloader ===")
print((repo / "data.py").read_text()[:9000])

print("\n=== train.py ===")
print((repo / "train.py").read_text()[:5000])

print("\n=== state_dicts directory ===")
weights_dir = repo / "cifar10_models" / "state_dicts"
print("Exists:", weights_dir.exists())
if weights_dir.exists():
    print([(p.name, p.stat().st_size) for p in weights_dir.iterdir()])

=== data.py: weight downloader ===
import os
import zipfile

import pytorch_lightning as pl
import requests
from torch.utils.data import DataLoader
from torchvision import transforms as T
from torchvision.datasets import CIFAR10
from tqdm import tqdm


class CIFAR10Data(pl.LightningDataModule):
    def __init__(self, args):
        super().__init__()
        self.hparams = args
        self.mean = (0.4914, 0.4822, 0.4465)
        self.std = (0.2471, 0.2435, 0.2616)

    def download_weights():
        url = (
            "https://rutgers.box.com/shared/static/gkw08ecs797j2et1ksmbg1w5t3idf5r5.zip"
        )

        # Streaming, so we can iterate over the response.
        r = requests.get(url, stream=True)

        # Total size in Mebibyte
        total_size = int(r.headers.get("content-length", 0))
        block_size = 2 ** 20  # Mebibyte
        t = tqdm(total=total_size, unit="MiB", unit_scale=True)

        with open("state_dicts.zip", "wb") as f:
            for data in r.iter_con

### A.4 Confirm the Box.com link is dead
The response is an HTML 404 page (no `PK` ZIP signature), which explains the `BadZipFile` error above.

In [ ]:
import requests
from pathlib import Path

url = "https://rutgers.box.com/shared/static/gkw08ecs797j2et1ksmbg1w5t3idf5r5.zip"

r = requests.get(url, stream=True, allow_redirects=True, timeout=60)

print("HTTP status:", r.status_code)
print("Final URL:", r.url)
print("Content-Type:", r.headers.get("Content-Type"))
print("Content-Length:", r.headers.get("Content-Length"))

# Inspect the first bytes, without downloading the whole file
first_bytes = next(r.iter_content(chunk_size=32), b"")
print("First bytes:", repr(first_bytes))
print("ZIP signature:", first_bytes.startswith(b"PK"))

r.close()

HTTP status: 404
Final URL: https://rutgers.app.box.com/public/static/gkw08ecs797j2et1ksmbg1w5t3idf5r5.zip
Content-Type: text/html; charset=utf-8
Content-Length: None
First bytes: b'\n <!DOCTYPE html><html lang="en-'
ZIP signature: False